# Capstone — mirrors your deployed research paper

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

**What performance archetypes exist across a content inventory, and how should a content team
prioritize action on them?**

This supports one decision: which pages a content strategist or editor should review first in a
given sprint — protect, improve, rewrite, monitor — using only structured, safe metrics (traffic,
engagement, freshness, token counts). No article text was used, so this is behavioral clustering,
not semantic clustering. The work is decision-support for a human reviewer, not an automated
content system.

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

## Data

**Source:** FlyRank ML Internship starter dataset — `data/raw/content_refresh_anonymized.csv`,
30,000 rows, one row per pseudonymized content page, 32 clients, trailing-90-day metrics.

**Date window:** trailing 90 days from extract date, with layered sub-windows (`*_last_30d`,
`*_prev_30d` for trend calculation).

**Excluded fields, with why:**
- `trend_direction` / `trend_pct` as ML **features** (though used as a rule input for the baseline)
  — these are derived the same way a label would be, so using them as model inputs would be circular.
- `provider_used` / `model_used` — production/tooling metadata, not performance signal; mixing it in
  risks clusters reflecting the content pipeline rather than audience behavior.
- `content_id` / `client_id` — identifiers, used only for grouping/splitting, never as features.
- No client names, raw URLs, or query strings appear anywhere in this dataset or this paper —
  all identifiers are pseudonyms per the dataset's own design.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

In [2]:
print(f"Rows: {len(df)} | Unique clients: {df['client_id'].nunique()} | Unique pages: {df['content_id'].nunique()}")

Rows: 30000 | Unique clients: 32 | Unique pages: 30000


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

**Task type:** Clustering (unsupervised) — "what kinds of pages exist," not "will this page
decline." No observed future outcome exists in this snapshot, so classification was not honest here.

**Label:** none. Archetype names (champion, hidden gem, weak/no-demand) are assigned to clusters
*after* inspecting their contents, not defined in advance.

**Baseline:** a transparent, human-readable rule (`stale_declining_visible`,
`declining_visible_fresh`, `stale_visible_stable`, `low_priority`), scored as
`declining × visible × (1 + stale) × impressions_90d`, with one reason code per page.

**Features:** structured 90-day metrics (search economics, performance, engagement, freshness) plus
`has_-flags` for four fields confirmed to be missing by `content_type` rather than randomly
(`search_volume`, `competition`, `cpc`, `word_count`).

**Validation design:** grouped split by `client_id` (`GroupShuffleSplit`), not random — client sizes
are highly skewed (largest client = 23% of rows), so a random split lets the model memorize a
client's own scale rather than generalize.

**Leakage checks:** full checklist run (label-derived fields, product-decision flags, ID columns,
group overlap) plus a deliberate-injection test — adding a known-leaky field
(`trend_pct`) measurably raised holdout silhouette (0.278 → 0.293), confirming the audit harness
was actually sensitive to leakage rather than rubber-stamping a clean-looking list.

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

In [3]:
comparison_table = pd.DataFrame({
    "Metric": [
        "Groups found",
        "Resolution inside baseline's largest bucket (21,085 rows)",
        "Holdout silhouette — random split",
        "Holdout silhouette — grouped-by-client split (honest)",
        "Holdout silhouette — with deliberately leaked feature",
    ],
    "Baseline (4 hand-written buckets)": ["4", "1 undifferentiated bucket", "—", "—", "—"],
    "K-Means (k=4)": ["4", "4 distinct sub-clusters", "0.294", "0.278", "0.293 (confirms leakage detection works)"],
})
print(comparison_table.to_string(index=False))

                                                   Metric Baseline (4 hand-written buckets)                            K-Means (k=4)
                                             Groups found                                 4                                        4
Resolution inside baseline's largest bucket (21,085 rows)         1 undifferentiated bucket                  4 distinct sub-clusters
                        Holdout silhouette — random split                                 —                                    0.294
    Holdout silhouette — grouped-by-client split (honest)                                 —                                    0.278
    Holdout silhouette — with deliberately leaked feature                                 — 0.293 (confirms leakage detection works)


K-Means found real but partial structure beyond the baseline's four buckets — most clearly a
382-page high-performing cluster (median 503 sessions, 2.63% engagement, page-1 position) and a
2,469-page near-invisible cluster (median 6 impressions). However, a shallow decision tree reading
the clusters back showed the split leaned heavily on `has_word_count` (importance 0.271) — a
near-proxy for `content_type` — meaning part of the clustering rediscovers known content categories
rather than finding a wholly new behavioral pattern. This is stated as a limitation, not smoothed over.

## 5. Limitations

*What this work cannot claim.*

- **No article text** — clustering is behavioral/structural, not semantic. Archetype names describe
  traffic and engagement shape, not topic or meaning.
- **Content-type confound** — the strongest driver of cluster split (`has_word_count`) is a near-proxy
  for `content_type`, so part of the "archetype" signal is closer to content category than genuine
  behavioral discovery.
- **No validated future outcome** — the baseline rule uses `trend_direction`, itself a defined rule
  (30d vs. prev 30d), not an independently observed future result. No precision@K against a real
  future window exists in this snapshot.
- **Client concentration** — one client holds 23% of rows; the priority queue's top candidates are
  disproportionately drawn from a small number of large clients, confirmed directly rather than assumed.
- **Small holdout for grouped validation** — the grouped split's holdout covers only 7 of 32 clients;
  a single unusual client could swing the reported silhouette more than a larger holdout would.
- **Cross-sectional, one snapshot** — no causal claims are supported. This is observed, directional,
  decision-support structure only.

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

In [6]:
import os
print("cwd:", os.getcwd())
print("work/ exists:", os.path.isdir("work"))
print("work/outputs/ exists:", os.path.isdir("work/outputs"))
if os.path.isdir("work/outputs"):
    print("contents:", os.listdir("work/outputs"))

cwd: /content/flyrank-ml-internship-starter
work/ exists: True
work/outputs/ exists: False


In [7]:
os.makedirs("work/outputs", exist_ok=True)
# re-run the ranked_queue build from ML-10 section 1, then:
export_cols = [
    "content_id", "client_id", "content_type", "action", "reason_code", "archetype",
    "priority_score", "impressions_90d", "sessions_90d", "trend_direction", "trend_pct",
    "avg_position", "engagement_rate", "days_since_last_update", "content_age_days", "word_count"
]
ranked_queue[export_cols].to_csv("work/outputs/content_action_playbook.csv", index=False)
print("Regenerated:", os.path.exists("work/outputs/content_action_playbook.csv"))

NameError: name 'ranked_queue' is not defined

In [4]:
# pulled directly from the ML-10 playbook export
playbook = pd.read_csv("work/outputs/content_action_playbook.csv")
print(playbook["action"].value_counts())
print(playbook.groupby("action", observed=True)[["impressions_90d", "priority_score"]].median())

FileNotFoundError: [Errno 2] No such file or directory: 'work/outputs/content_action_playbook.csv'

1. **Protect and refresh champions first** (3,015 pages) — highest-value pages, often also going
   stale; losing these costs the most.
2. **Rewrite the small, high-confidence stale-and-declining group** (13 pages) — still visible,
   losing ground, untouched 180+ days.
3. **Improve the larger declining-but-fresh group** (14,907 pages) — investigate *why* a recently
   updated page is still declining before assuming a rewrite fixes it.
4. **Monitor the rest** (12,065 pages) — includes weak/no-demand pages where `avg_position` is
   statistically unreliable due to tiny impression bases; do not prune on position alone.

No action here should be automated end-to-end — see the human-review and no-go list in the ML-10
notebook for what a person must check before acting on any recommendation.

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

In [5]:
import matplotlib.pyplot as plt

# Chart 1: action queue distribution
action_counts = playbook["action"].value_counts()
fig, ax = plt.subplots(figsize=(6,4))
action_counts.plot(kind="bar", ax=ax, color="#2F6F6B")
ax.set_title("Pages per recommended action")
ax.set_ylabel("Page count")
plt.tight_layout()
plt.savefig("work/outputs/fig_action_distribution.png", dpi=150)
plt.close()

# Chart 2: baseline vs clustering resolution inside low_priority
comparison_table.to_csv("work/outputs/table_model_vs_baseline.csv", index=False)

print("Saved: fig_action_distribution.png, table_model_vs_baseline.csv")

NameError: name 'playbook' is not defined

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.